# Topic Modeling Ablation: BERTopic vs. LDA vs. LLM Baselines
This notebook implements a robust ablation study to compare the quality and stability of BERTopic against classical (LDA) and LLM-based topic modeling approaches.

**Experiments:**
1. **Topic Quality**: Evaluate NPMI and Topic Diversity at K=50.
2. **Stability**: Measure Adjusted Rand Index (ARI) and Jaccard Similarity across 3 independent runs.


In [2]:
import os
import pandas as pd
import numpy as np
import concurrent.futures
from tqdm.notebook import tqdm
from openai import OpenAI
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import LatentDirichletAllocation
from bertopic import BERTopic
import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel
import random
from dotenv import load_dotenv

# For reproducibility in sampling
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Load API key from .env
load_dotenv('../.env')
client = OpenAI(api_key=os.getenv('DLAB_OPENAI_KEY'))


In [3]:
# 1. Load Data & Subsample
df = pd.read_csv('../data/processed/posts_w_labels.csv')

# Drop NA selftexts
df = df.dropna(subset=['selftext'])
docs = df['selftext'].tolist()

print(f"Loaded {len(docs)} documents for ablation.")


Loaded 12747 documents for ablation.


In [4]:
# --- Evaluation Functions ---

def calculate_topic_diversity(topic_words_list, top_k=25):
    """
    Calculates Topic Diversity: Percentage of unique words across all topics' top K words.
    """
    all_words = []
    for words in topic_words_list:
        all_words.extend(words[:top_k])
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words) if all_words else 0
    return diversity

def calculate_npmi(docs, topic_words_list, top_k=10):
    """
    Calculates NPMI (Coherence) using Gensim.
    """
    texts = [doc.split() for doc in docs]
    id2word = corpora.Dictionary(texts)
    
    # gensim expects list of list of strings
    topics_for_gensim = [words[:top_k] for words in topic_words_list]
    
    cm = CoherenceModel(topics=topics_for_gensim, texts=texts, dictionary=id2word, coherence='c_npmi')
    return cm.get_coherence()

def calculate_stability_ari(labels_run1, labels_run2, labels_run3):
    """
    Calculates the average Adjusted Rand Index (ARI) across 3 runs.
    """
    ari_12 = adjusted_rand_score(labels_run1, labels_run2)
    ari_13 = adjusted_rand_score(labels_run1, labels_run3)
    ari_23 = adjusted_rand_score(labels_run2, labels_run3)
    return np.mean([ari_12, ari_13, ari_23])


In [5]:
# --- Model A: BERTopic (3 Runs) ---
# We force nr_topics=50 to match the evaluation constraints

bertopic_runs_labels = []
bertopic_runs_topics = []

for i, seed in enumerate([42, 123, 999]):
    print(f"Running BERTopic Seed {seed}...")
    
    # Initialize UMAP with different seed for stability testing
    from umap import UMAP
    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=seed)
    
    topic_model = BERTopic(nr_topics=50, umap_model=umap_model)
    topics, probs = topic_model.fit_transform(docs)
    
    bertopic_runs_labels.append(topics)
    
    # Extract top words (ignoring outlier topic -1 if present for diversity calculation)
    topic_info = topic_model.get_topic_info()
    top_words = []
    for topic_id in topic_info['Topic']:
        if topic_id != -1:
            words = [word for word, _ in topic_model.get_topic(topic_id)]
            top_words.append(words)
    
    bertopic_runs_topics.append(top_words)

print("BERTopic runs complete.")


Running BERTopic Seed 42...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Running BERTopic Seed 123...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Running BERTopic Seed 999...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BERTopic runs complete.


In [6]:
# --- Model B: LDA (3 Runs) ---
# We use K=50

vectorizer = CountVectorizer(stop_words='english', max_features=5000)
X = vectorizer.fit_transform(docs)
feature_names = vectorizer.get_feature_names_out()

lda_runs_labels = []
lda_runs_topics = []

for i, seed in enumerate([42, 123, 999]):
    print(f"Running LDA Seed {seed}...")
    
    lda = LatentDirichletAllocation(n_components=50, random_state=seed, max_iter=10)
    doc_topic_dist = lda.fit_transform(X)
    
    # Document assignments (argmax of distribution)
    labels = np.argmax(doc_topic_dist, axis=1)
    lda_runs_labels.append(labels)
    
    # Extract top words
    top_words = []
    for topic_idx, topic in enumerate(lda.components_):
        top_features_ind = topic.argsort()[:-26:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        top_words.append(top_features)
        
    lda_runs_topics.append(top_words)

print("LDA runs complete.")


Running LDA Seed 42...
Running LDA Seed 123...
Running LDA Seed 999...
LDA runs complete.


In [7]:
# --- Model C: LLM Baseline ---
def get_topics_for_doc(doc, seed):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini-2024-07-18",
            messages=[
                {"role": "system", "content": "You are a topic extraction tool. Extract the single most prominent high-level topic from the text. Respond with ONLY 1-3 words. No punctuation."},
                {"role": "user", "content": doc[:2000]} # Truncating to save tokens/time
            ],
            temperature=0.7,
            seed=seed,
            max_tokens=10
        )
        return response.choices[0].message.content.strip().lower()
    except Exception as e:
        return "unknown"

def run_llm_pipeline(docs, K=50, seed=42):
    print(f"  [1/3] Generating topics with GPT-4o-mini for {len(docs)} documents (Seed {seed})...")
    
    # 1. Topic Generation (LLM per document)
    # Using ThreadPoolExecutor to run API calls concurrently (otherwise 10k docs will take hours)
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        # map guarantees order is preserved
        raw_topics = list(tqdm(executor.map(lambda d: get_topics_for_doc(d, seed), docs), total=len(docs)))
        
    print(f"  [2/3] Embedding raw topics and clustering into K={K}...")
    # 2. Topic Collapsing (WSM / Clustering)
    # Deduplicate for embedding to save money/time
    unique_topics = list(set(raw_topics))
    
    # Get embeddings for unique topics (Batched to avoid 2048 limit)
    embeddings_dict = {}
    batch_size = 2000
    for i in range(0, len(unique_topics), batch_size):
        batch = unique_topics[i:i + batch_size]
        res = client.embeddings.create(input=batch, model="text-embedding-3-small")
        for t, emb in zip(batch, res.data):
            embeddings_dict[t] = emb.embedding
    
    # Map back to full document list
    X_emb = np.array([embeddings_dict[t] for t in raw_topics])
    
    # Cluster the topic embeddings to collapse them into K topics
    kmeans = KMeans(n_clusters=K, random_state=seed, n_init='auto')
    labels = kmeans.fit_predict(X_emb)
    
    print(f"  [3/3] Computing c-TF-IDF / Frequency representations...")
    # 3. Topic Representation
    # Combine documents per cluster to compute top words (similar to BERTopic's c-TF-IDF step)
    docs_per_class = pd.DataFrame({'doc': docs, 'class': labels}).groupby('class')['doc'].apply(' '.join).reset_index()
    
    vectorizer = CountVectorizer(stop_words='english', max_features=5000)
    X_counts = vectorizer.fit_transform(docs_per_class['doc'])
    words = vectorizer.get_feature_names_out()
    
    # Take the top 25 words by frequency in each cluster
    top_words = []
    for i in range(K):
        row = X_counts[i].toarray()[0]
        top_indices = row.argsort()[-25:][::-1]
        top_words.append([words[idx] for idx in top_indices])
        
    return labels, top_words

llm_runs_labels = []
llm_runs_topics = []

print("Starting LLM Baseline Runs...")
# WARNING: Running this 3 times on 10k documents is 30,000 API calls! 
for seed in [42, 123, 999]:
    print(f"\n--- Running LLM Pipeline Seed {seed} ---")
    labels, topics = run_llm_pipeline(docs, K=50, seed=seed)
    llm_runs_labels.append(labels)
    llm_runs_topics.append(topics)

print("\nLLM runs complete.")


Starting LLM Baseline Runs...

--- Running LLM Pipeline Seed 42 ---
  [1/3] Generating topics with GPT-4o-mini for 12747 documents (Seed 42)...


  0%|          | 0/12747 [00:00<?, ?it/s]

  [2/3] Embedding raw topics and clustering into K=50...
  [3/3] Computing c-TF-IDF / Frequency representations...

--- Running LLM Pipeline Seed 123 ---
  [1/3] Generating topics with GPT-4o-mini for 12747 documents (Seed 123)...


  0%|          | 0/12747 [00:00<?, ?it/s]

  [2/3] Embedding raw topics and clustering into K=50...
  [3/3] Computing c-TF-IDF / Frequency representations...

--- Running LLM Pipeline Seed 999 ---
  [1/3] Generating topics with GPT-4o-mini for 12747 documents (Seed 999)...


  0%|          | 0/12747 [00:00<?, ?it/s]

  [2/3] Embedding raw topics and clustering into K=50...
  [3/3] Computing c-TF-IDF / Frequency representations...

LLM runs complete.


In [10]:
# --- Final Evaluation ---
# 1. Quality (using Run 1 for each model)
bertopic_div = calculate_topic_diversity(bertopic_runs_topics[0])
lda_div = calculate_topic_diversity(lda_runs_topics[0])
llm_div = calculate_topic_diversity(llm_runs_topics[0])

print("--- Topic Quality ---")
print(f"BERTopic Diversity: {bertopic_div:.3f}")
print(f"LDA Diversity:      {lda_div:.3f}")
print(f"LLM Diversity:      {llm_div:.3f}")
print()

bertopic_npmi = calculate_npmi(docs, bertopic_runs_topics[0])
lda_npmi = calculate_npmi(docs, lda_runs_topics[0])
llm_npmi = calculate_npmi(docs, llm_runs_topics[0])
print(f"BERTopic Coherence: {bertopic_npmi:.3f}")
print(f"LDA Coherence:      {lda_npmi:.3f}")
print(f"LLM Coherence:      {llm_npmi:.3f}")

# 2. Stability
bertopic_ari = calculate_stability_ari(*bertopic_runs_labels)
lda_ari = calculate_stability_ari(*lda_runs_labels)
llm_ari = calculate_stability_ari(*llm_runs_labels)

print("\n--- Topic Stability (ARI) ---")
print(f"BERTopic ARI: {bertopic_ari:.3f}")
print(f"LDA ARI:      {lda_ari:.3f}")
print(f"LLM ARI:      {llm_ari:.3f}")


--- Topic Quality ---
BERTopic Diversity: 0.700
LDA Diversity:      0.317
LLM Diversity:      0.092

BERTopic Coherence: -0.030
LDA Coherence:      -0.064
LLM Coherence:      -0.136

--- Topic Stability (ARI) ---
BERTopic ARI: 0.873
LDA ARI:      0.135
LLM ARI:      0.533
